# Experiment 9: when does the model realise a front is coming?

Meteorologists compare successive forecasts for the **same target hour** ("dprog/dt", e.g. Hamill 2003): how does
the forecast for one moment change as that moment gets closer?

Data: the runs saved by Experiment 7 (no new model run): 5 days around the 25-26 March front, all 378 points, one
run every hour, each run 24 h long (temperature only, 10 weeks of history).

**Hours ahead** = target hour minus the last hour the model knew (1 = the very next hour).
**Map error** = root mean square difference between forecast (median) and what happened over all points (°C).

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp9")
FIG.mkdir(parents=True, exist_ok=True)
EXP7 = Path("/workspace/data/notebooks/results/exp7_time_curves_378pts_1h.npz")
if os.environ.get("QUICK"):
    FIG, EXP7 = Path("/tmp"), Path("/tmp/exp7_test.npz")

z = np.load(EXP7)
lead7 = int(z["lead"])
runs = pd.to_datetime(z["times"]) - pd.Timedelta(hours=lead7)  # start hour of each run (first forecast hour)
pids = z["point_id"]
fq = z["forecast_all_leads"].astype(np.float32)  # (runs, points, q10/q50/q90, 24)

df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", "T2M"])
Tw = df.pivot(index="time", columns="point_id", values="T2M").interpolate(limit=3, limit_area="inside")[pids]
ev = pd.read_parquet(DATA / "events.parquet")
ev = ev[ev.kind == "cold_front"]

targets = pd.date_range(pd.to_datetime(z["times"])[0], pd.to_datetime(z["times"])[-1], freq="h")
run_idx = {t: i for i, t in enumerate(runs)}
E = np.full((24, len(targets)), np.nan)
for j, t in enumerate(targets):
    truth = Tw.loc[t].to_numpy()
    for k in range(1, 25):  # hours ahead
        r = run_idx.get(t - pd.Timedelta(hours=k - 1))
        if r is not None:
            E[k - 1, j] = np.sqrt(np.nanmean((fq[r, :, 1, k - 1] - truth) ** 2))
share = ev[ev.point_id.isin(pids)].groupby("time").point_id.nunique().reindex(targets).fillna(0).to_numpy() / len(pids)
print(f"{len(runs)} runs, {len(pids)} points, {len(targets)} target hours")

120 runs, 378 points, 120 target hours


## Error by target hour and hours ahead

Each column is one target hour, each row how many hours ahead the forecast for it was made. A front that the
model only "sees" late shows up as a bright column that turns dark only near the bottom (short lead times).
Each diagonal line from bottom-left to top-right is one single run. Top: share of points in a front passage.

In [2]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.2, 0.8], vertical_spacing=0.04)
fig.add_trace(go.Bar(x=targets, y=100 * share, marker_color="#1f77b4", name="points in front passage (%)",
                     hovertemplate="%{x|%a %d %b %H:%M}: %{y:.0f} %<extra></extra>"), row=1, col=1)
fig.add_trace(go.Heatmap(z=E, x=targets, y=np.arange(1, 25), colorscale="Magma", colorbar={"title": "map error (°C)", "y": 0.4, "len": 0.8},
                         hovertemplate="target %{x|%a %d %b %H:%M}, %{y} h ahead: %{z:.1f} °C<extra></extra>"), row=2, col=1)
fig.update_yaxes(title_text="in passage (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="hours ahead (1 = made 1 h before)", dtick=3, row=2, col=1)
fig.update_xaxes(tickformat="%a %d %b\n%H:%M", row=2, col=1)
fig.update_layout(height=620, template="plotly_dark", showlegend=False,
                  title="When does the model realise the front is coming?<br><sup>forecast map error for each target hour (x), "
                        "by how many hours ahead the forecast was made (y); 378 points, 25-26 March front</sup>")
fig.write_html(FIG / "exp9_error_by_lead.html", include_plotlyjs="cdn")
fig.show()

## The front hour vs a calm hour

Map error by hours ahead for the hour with the most points in the front passage, and for the same hour of day
two days earlier (calm).

In [3]:
jf = int(np.argmax(share))
jc = jf - 48 if jf >= 48 else None
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(1, 25), y=E[:, jf], mode="lines+markers", line={"color": "#d62728", "width": 3},
                         name=f"front hour ({targets[jf]:%a %d %b %H:%M})"))
if jc is not None:
    fig.add_trace(go.Scatter(x=np.arange(1, 25), y=E[:, jc], mode="lines+markers", line={"color": "#7f7f7f", "width": 3},
                             name=f"calm hour ({targets[jc]:%a %d %b %H:%M})"))
fig.update_layout(height=400, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title="How the forecast for one hour improves as it gets closer",
                  xaxis={"title": "hours ahead", "dtick": 2, "autorange": "reversed"}, yaxis={"title": "map error (°C)", "rangemode": "tozero"})
fig.write_html(FIG / "exp9_front_vs_calm.html", include_plotlyjs="cdn")
fig.show()

## One point: successive forecasts for the drop

The point with the steepest 6 h fall in the window. White = what happened. Coloured lines = forecasts started
24, 18, 12, 9, 6, 3 and 1 h before the drop (purple = early, yellow = late). When does a run first bend down?

In [4]:
Y = Tw.loc[targets].to_numpy()
fall = Y[:-6] - Y[6:]
jt, pc = np.unravel_index(np.nanargmax(fall), fall.shape)
td = targets[jt]
span = pd.date_range(td - pd.Timedelta("30h"), td + pd.Timedelta("30h"), freq="h")
KS = [24, 18, 12, 9, 6, 3, 1]
cols = sample_colorscale("Plasma", [0.15 + 0.8 * i / (len(KS) - 1) for i in range(len(KS))])
fig = go.Figure()
fig.add_trace(go.Scatter(x=span, y=Tw[pids[pc]].reindex(span), mode="lines", line={"color": "white", "width": 3.5}, name="what happened"))
for k, c in zip(KS, cols):
    r = run_idx.get(td - pd.Timedelta(hours=k - 1))
    if r is None:
        continue
    ft = pd.date_range(runs[r], periods=24, freq="h")
    fig.add_trace(go.Scatter(x=ft, y=fq[r, pc, 1], mode="lines", line={"color": c, "width": 2.5}, name=f"run {k} h before the drop",
                             hovertemplate="%{x|%a %d %b %H:%M}: %{y:.1f} °C<extra>" + f"{k} h before" + "</extra>"))
fig.add_vline(x=td, line={"color": "grey", "dash": "dot"})
fig.update_layout(height=450, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title=f"Point {pids[pc]}: successive forecasts for the drop of {fall[jt, pc]:.1f} °C in 6 h (from {td:%a %d %b %H:%M} UTC)",
                  xaxis={"tickformat": "%a %d %b\n%H:%M"}, yaxis={"title": "T2M (°C)"})
fig.write_html(FIG / "exp9_successive_runs.html", include_plotlyjs="cdn")
fig.show()